# param-group-dict-list — worked example 3: frozen backbone via a zero-lr group

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-group-dict-list`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A param group can carry any optimizer hyperparameter, including `lr=0.0` to effectively freeze a subset while still registering it with the optimizer. This keeps the backbone in the optimizer state without updating it.

## Worked solution

We implement `make_frozen_backbone_groups(backbone, head, head_lr)` returning two dicts: the backbone with `lr=0.0` and the head with `head_lr`. We build the groups into a real SGD optimizer, attach a gradient of ones to one backbone parameter and one head parameter, snapshot both, and call `optimizer.step()`. Because the backbone group has zero learning rate, its parameter does not move, while the head parameter moves by `-head_lr`. We print the change magnitude for each to show the backbone stayed put.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

def make_frozen_backbone_groups(backbone, head, head_lr):
    return [
        {'params': list(backbone.parameters()), 'lr': 0.0},
        {'params': list(head.parameters()), 'lr': head_lr},
    ]

backbone = nn.Linear(4, 4)
head = nn.Linear(4, 1)
opt = t.optim.SGD(make_frozen_backbone_groups(backbone, head, 0.1))
for p in list(backbone.parameters()) + list(head.parameters()):
    p.grad = t.ones_like(p)
bb0 = backbone.weight.data.clone()
hd0 = head.weight.data.clone()
opt.step()
print('backbone moved:', (backbone.weight.data - bb0).abs().sum().item())
print('head moved:', (head.weight.data - hd0).abs().sum().item())